# Student Dropout Prediction — Integrated Analysis

This notebook consolidates exploratory analysis, preprocessing, initial model comparison, tuning, ensembles, feature-group comparisons, and permutation importance. It replaces repeated setup with one dataset and one stratified split. Original source files remain available in the repository.

**Scope:** development on the 3,539 training records only; the 885 test records are reserved. Final test evaluation is intentionally not included. Output cells are cleared because this consolidated version has not been executed end to end. Previously saved results must not be represented as outputs of this new notebook.

Run cells in order with Python, pandas, NumPy, SciPy, scikit-learn, XGBoost, Matplotlib, seaborn (0.13 or later), IPython, and tabulate. Place the repository dataset ZIP or `data.csv` beside this notebook or in its parent directory. Outputs go to `combined_results/`, preserving the original result filenames in separate subdirectories. Exact historical dependency versions are unavailable; reruns may differ.


## 1. Imports and settings


In [ ]:
from pathlib import Path
import io, json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import (StratifiedKFold, RandomizedSearchCV,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, LabelEncoder,
                                   OneHotEncoder, StandardScaler)
from xgboost import XGBClassifier

RANDOM_STATE = 42
FAST_MODE = False  # Set True for a quick practice run; set False for final CV/plots.
N_SPLITS = 5  # Five folds in both full and quick modes.
N_PERM_REPEATS = 2 if FAST_MODE else 4
ITERATIONS = {"XGBoost": 3 if FAST_MODE else 10,
              "Gradient Boosting": 3 if FAST_MODE else 10,
              "Logistic Regression": 3 if FAST_MODE else 8}
CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
RESULTS_ROOT = Path('combined_results').resolve()
EDA_DIR = RESULTS_ROOT / 'eda'
OUTPUT_DIR = RESULTS_ROOT / 'part_e_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


import hashlib
import matplotlib
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 2. Read data once and reserve the test set

The split uses the original row order, stratification on Target, and seed 42. The original CSV bytes are hashed. The test set is retained for subsequent final evaluation but is not scored here.


In [ ]:
# Search both normal working locations, no absolute personal machine paths.
search_dirs = list(dict.fromkeys([Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
                                  Path.cwd() / 'insy674_partb_result']))
zip_candidates = []
csv_candidates = []
for parent in search_dirs:
    if parent.exists():
        zip_candidates.extend(sorted(parent.glob('predict*students*dropout*.zip')))
        csv_candidates.extend([parent / 'data.csv'])

def load_dataset():
    for path in zip_candidates:
        if not path.is_file():
            continue
        with zipfile.ZipFile(path) as archive:
            names = [n for n in archive.namelist() if Path(n).name.lower() == 'data.csv']
            if names:
                raw_bytes = archive.read(names[0])
                data = pd.read_csv(io.BytesIO(raw_bytes), sep=';', encoding='utf-8-sig')
                return data, f'{path} -> {names[0]}', raw_bytes
    for path in csv_candidates:
        if path.is_file():
            return pd.read_csv(path, sep=';', encoding='utf-8-sig'), str(path), path.read_bytes()
    raise FileNotFoundError('Cannot locate the repository dataset ZIP or data.csv. Run from the repo root or put data.csv beside this notebook.')

df, DATA_SOURCE, DATA_BYTES = load_dataset()
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
assert 'Target' in df.columns, 'Expected a Target column.'
assert set(df['Target'].unique()) == {'Dropout', 'Enrolled', 'Graduate'}
print('Data:', DATA_SOURCE, '| shape:', df.shape)
print('Labels:', df['Target'].value_counts().to_dict())
row_ids = np.arange(len(df))
train_ids, test_ids = train_test_split(row_ids, test_size=0.20,
    random_state=RANDOM_STATE, stratify=df['Target'])
X = df.drop(columns=['Target'])
y = df['Target']
X_train, X_test = X.iloc[train_ids].copy(), X.iloc[test_ids].copy()
y_train, y_test = y.iloc[train_ids].copy(), y.iloc[test_ids].copy()
assert not set(train_ids).intersection(test_ids)
assert len(df) == 4424 and len(train_ids) == 3539 and len(test_ids) == 885
print('Training:', len(train_ids), '| Reserved test:', len(test_ids))


## 3. Exploratory data analysis

All outcome-related summaries and six figures use training rows. Full-file duplicate checks are structural checks. Second-semester values are excluded from substantive EDA. Macroeconomic variables are descriptive context only; later models exclude them.


In [ ]:
ROOT = EDA_DIR
ROOT.mkdir(parents=True, exist_ok=True)
for folder in ["figures", "tables", "reports"]:
    (ROOT / folder).mkdir(exist_ok=True)
ORDER = ['Dropout', 'Enrolled', 'Graduate']
COLORS = ['#C65B51', '#D5A541', '#32818A']
FIN = ['Scholarship holder', 'Debtor', 'Tuition fees up to date']
CAT = ['Marital status', 'Application mode', 'Course', 'Daytime/evening attendance',
       'Previous qualification', 'Nacionality', "Mother's qualification", "Father's qualification",
       "Mother's occupation", "Father's occupation", 'Displaced', 'Educational special needs',
       'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']
NUM = ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
       'Unemployment rate', 'Inflation rate', 'GDP', 'Application order']
SOURCE = 'https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success'

def md(df):
    def clean(x): return str(x).replace('|', '/').replace('\n', ' ')
    return '| ' + ' | '.join(map(clean, df.columns)) + ' |\n|' + '|'.join(['---']*len(df.columns)) + '|\n' + '\n'.join('| ' + ' | '.join(map(clean,row)) + ' |' for row in df.itertuples(index=False,name=None))

def save(name):
    plt.savefig(ROOT/'figures'/f'{name}.png', dpi=180, bbox_inches='tight')
    display(fig := plt.gcf())
    plt.close(fig)


In [ ]:
d = df.copy()
ids = row_ids
tr, te = train_ids, test_ids
# Export assignment only: original CSV row positions are identifiers, not predictors.
assignment=pd.DataFrame({'row_id':ids,'split':np.where(np.isin(ids,tr),'train','test')})
assignment.to_csv(ROOT/'tables/split_assignments.csv',index=False)
x=d.iloc[tr].copy()
features=[c for c in d if c!='Target']
second=[c for c in features if '2nd sem' in c]
first=[c for c in features if '1st sem' in c]
eligible=[c for c in features if c not in second]
numeric=[c for c in eligible if c not in CAT]
sns.set_theme(style='whitegrid',font_scale=1.05)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.figsize':(9,5)})
counts=x.Target.value_counts().reindex(ORDER)
ct=pd.DataFrame({'Status':ORDER,'n':counts.values,'Percent':(100*counts/len(x)).round(2).values})
ct.to_csv(ROOT/'tables/class_distribution.csv',index=False)
fig,ax=plt.subplots()
bars=ax.bar(ORDER,counts,color=COLORS)
for b,n in zip(bars,counts): ax.text(b.get_x()+b.get_width()/2,n+15,f'{n:,} ({n/len(x):.1%})',ha='center')
ax.set(title='Training outcomes are imbalanced',ylabel='Number of students',ylim=(0,max(counts)*1.17))
ax.text(0,-.17,f'Training set only, n = {len(x):,}',transform=ax.transAxes,color='#555555')
save('01_target_distribution')
# All attributes profiled; second-semester values deliberately not explored.
rows=[]
for c in features:
    excluded=c in second
    s=x[c]
    role='Financial' if c in FIN else ('Academic' if c in first or c in ['Admission grade','Previous qualification','Previous qualification (grade)'] else 'Context')
    timing='Excluded: after prediction horizon' if excluded else ('End of first semester' if c in first else 'Enrollment candidate; verify administrative timing')
    note='Do not interpret numeric codes as continuous.' if c in CAT else 'Inspect skew and extremes; retain valid observations.'
    if c in ["Mother's qualification", "Father's qualification"]:
        note += f' Unknown code 34: {int((s==34).sum())} ({(s==34).mean():.1%}).'
    if '(grade)' in c and 'sem' in c: note+=' Zero grade may indicate no successful assessment; do not replace with missing automatically.'
    if c in FIN: note+=' Recording date unconfirmed; early-availability assumption must be documented.'
    rows.append({'feature':c,'group':role,'semantic_type':'Categorical code' if c in CAT else ('Count' if c in first or c in second else 'Numeric/ordinal'),
      'timing':timing,'missing_pct':round(s.isna().mean()*100,3) if not excluded else 'not analyzed',
      'unique':s.nunique() if not excluded else 'not analyzed',
      'distribution':('Excluded' if excluded else ('Category frequencies in categorical_counts.csv' if c in CAT else f'min={s.min():.3g}; median={s.median():.3g}; max={s.max():.3g}; skew={s.skew():.2f}')),
      'usefulness_and_preparation':('Excluded from both scenarios.' if excluded else note)})
profile=pd.DataFrame(rows)
profile.to_csv(ROOT/'tables/attribute_profile.csv',index=False)
pd.concat([x[c].value_counts(dropna=False).rename_axis('value').reset_index(name='n').assign(feature=c) for c in CAT],ignore_index=True).to_csv(ROOT/'tables/categorical_counts.csv',index=False)
x[numeric].describe().T.to_csv(ROOT/'tables/numeric_summary.csv')
fig,axes=plt.subplots(1,3,figsize=(14,4.5))
ac=['Admission grade','Curricular units 1st sem (grade)','Curricular units 1st sem (approved)']
labels=['Admission grade','First-semester grade','First-semester units approved']
for ax,c,label in zip(axes,ac,labels):
    sns.boxplot(data=x,x='Target',y=c,order=ORDER,hue='Target',hue_order=ORDER,palette=COLORS,legend=False,ax=ax,fliersize=2)
    ax.set(xlabel='',ylabel=label,title=label)
    ax.tick_params(axis='x',rotation=15)
fig.suptitle('Academic distributions by observed outcome (training set)',y=1.03)
fig.tight_layout(); save('02_academic_by_outcome')
med=x.groupby('Target')[ac].median().reindex(ORDER)
med.to_csv(ROOT/'tables/academic_medians.csv')
# Within-group rates; Wilson intervals for descriptive uncertainty.
financial=[]
fig,axes=plt.subplots(1,3,figsize=(14,4.5),sharey=True)
for ax,c in zip(axes,FIN):
    g=pd.crosstab(x[c],x.Target).reindex(columns=ORDER,fill_value=0)
    rates=g.div(g.sum(axis=1),axis=0)
    rates.plot.bar(stacked=True,color=COLORS,ax=ax,legend=False,rot=0)
    for j,(v,row) in enumerate(g.iterrows()):
        n=int(row.sum()); k=int(row['Dropout']); p=k/n; z=1.96
        center=(p+z*z/(2*n))/(1+z*z/n)
        half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
        financial.append({'feature':c,'value':v,'n':n,'dropout_n':k,'dropout_pct':round(100*p,2),'wilson_low_pct':round(100*(center-half),2),'wilson_high_pct':round(100*(center+half),2)})
        ax.text(j,1.025,f'n={n:,}',ha='center',fontsize=10)
    ax.set(title=c,xlabel='0 = No; 1 = Yes',ylim=(0,1.15))
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
axes[0].set_ylabel('Within-group share')
fig.legend(ORDER,loc='lower center',ncol=3,bbox_to_anchor=(.5,-.08))
fig.tight_layout(); save('03_financial_outcome_rates')
fin=pd.DataFrame(financial); fin.to_csv(ROOT/'tables/financial_dropout_rates.csv',index=False)
# Engineered indicator is exploratory; Implement it inside production pipeline.
enrolled=x['Curricular units 1st sem (enrolled)']
x['First-semester approval rate']=x['Curricular units 1st sem (approved)'].div(enrolled.where(enrolled>0))
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,c,label in zip(axes,['Age at enrollment','First-semester approval rate'],['Age at enrollment','Approved / enrolled units (first semester)']):
    sns.boxplot(data=x,x='Target',y=c,order=ORDER,hue='Target',hue_order=ORDER,palette=COLORS,legend=False,ax=ax,fliersize=2)
    ax.set(xlabel='',ylabel=label)
fig.tight_layout();save('04_age_and_approval_rate')
courses=x.groupby('Course').agg(n=('Target','size'),dropout_rate=('Target',lambda s:s.eq('Dropout').mean())).sort_values('n',ascending=False)
courses.to_csv(ROOT/'tables/course_dropout_rates.csv')
hist=['Age at enrollment','Admission grade','Curricular units 1st sem (grade)','Curricular units 1st sem (approved)']
fig,axes=plt.subplots(2,2,figsize=(11,7))
for ax,c in zip(axes.flat,hist): sns.histplot(data=x,x=c,bins=25,ax=ax,color=COLORS[2])
fig.tight_layout();save('05_numeric_distributions')
corrcols=['Admission grade','Previous qualification (grade)','Age at enrollment',*first,'Unemployment rate','Inflation rate','GDP']
corr=x[corrcols].corr(method='spearman')
corr.to_csv(ROOT/'tables/spearman_correlations.csv')
short=['Admission','Previous grade','Age','S1 credited','S1 enrolled','S1 evaluations','S1 approved','S1 grade','S1 no evaluations','Unemployment','Inflation','GDP']
fig,ax=plt.subplots(figsize=(11,9))
sns.heatmap(corr,xticklabels=short,yticklabels=short,cmap='vlag',vmin=-1,vmax=1,center=0,ax=ax,square=True,cbar_kws={'label':'Spearman correlation'})
ax.set_title('Numeric feature relationships (training set)')
save('06_numeric_correlations')
pairs=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().rename('rho').reset_index()
pairs.columns=['feature_1','feature_2','rho']; pairs['abs_rho']=pairs.rho.abs()
pairs=pairs.sort_values('abs_rho',ascending=False);pairs.to_csv(ROOT/'tables/top_correlations.csv',index=False)
metadata={'source':SOURCE,'data_sha256':hashlib.sha256(DATA_BYTES).hexdigest(),'total_rows':len(d),'train_n':len(x),'test_n':len(te),'seed':RANDOM_STATE,'test_size':0.2,'row_id':'zero-based original data row, excluding header','full_file_duplicate_rows':int(d.duplicated().sum()),'training_missing_cells':int(x[eligible].isna().sum().sum()),'second_semester_excluded':second}
(ROOT/'tables/run_metadata.json').write_text(json.dumps(metadata,indent=2))
rate_lines=[]
for c in FIN:
    f=fin[fin.feature.eq(c)].set_index('value')
    rate_lines.append(f'- **{c}:** dropout share is {f.loc[0,"dropout_pct"]:.1f}% for No (n={int(f.loc[0,"n"]):,}) and {f.loc[1,"dropout_pct"]:.1f}% for Yes (n={int(f.loc[1,"n"]):,}).')
report=f'''# Exploratory Data Analysis Report

## 1. Purpose and research alignment
This EDA examines outcome imbalance, academic patterns, financial indicators and data quality. It supports the team's questions about early prediction and the incremental predictive value of academic and financial information. It does not train a classifier or rank feature groups by predictive performance.

## 2. Source, scope and reproducibility
The UCI dataset contains {len(d):,} student records, 36 input variables and Target. The three labels describe status at the end of the normal course duration. Enrolled is an unresolved eventual outcome, not a guarantee of future graduation.

A stratified {1-0.2:.0%}/{0.2:.0%} split with random_state={RANDOM_STATE} assigns **{len(x):,} records to training and {len(te):,} to held-out testing**. All outcome-related figures and summaries below use training records only. No held-out performance is calculated. The input file structure and full-row duplicates are checked globally; these checks do not select predictors. `tables/split_assignments.csv` stores the shared row assignment and `run_metadata.json` records the source hash. All downstream modeling cells reuse this split.

Six second-semester fields are excluded from substantive exploration. The first-semester scenario includes earlier information plus six first-semester measurements. Availability dates of financial and administrative information remain unverified: results involving these fields are descriptive, conditional on later confirming availability.

## 3. Outcome distribution
{md(ct)}

![Training target distribution](../figures/01_target_distribution.png)

**Takeaway:** class frequencies differ. An always-Graduate classifier would have training accuracy of {counts['Graduate']/len(x):.1%}, without identifying any Dropout cases. This is a descriptive baseline, not a held-out model result. Model comparison should prioritize macro F1 and class-specific dropout recall/precision over accuracy alone.

## 4. Data quality and complete attribute profiling
No literal missing cells were found in the eligible training fields. The full CSV contains {int(d.duplicated().sum())} completely duplicated rows. These checks do not prove every value is valid or semantically complete. Parent qualification code 34 denotes Unknown in UCI documentation; preserve that distinction from a blank value. Other codes must be checked against the complete dictionary rather than assumed ordinal.

`tables/attribute_profile.csv` covers all 36 features: semantic type, feature group, time scope, missingness, cardinality, numeric range/skew or frequency-table reference, and preparation notes. Second-semester features are documented as excluded rather than having their values explored. Detailed category frequencies and numeric summaries accompany the profile.

The source header contains trailing whitespace in Daytime/evening attendance; the script strips header whitespace. Numeric codes for course, nationality, occupations and qualifications represent categories. Avoid Pearson correlations or continuous scaling of these codes as if the numeric distances were meaningful. Valid extreme ages or academic counts should not be dropped solely because they are boxplot outliers.

## 5. Academic factors
![Academic distributions](../figures/02_academic_by_outcome.png)

Training-group medians:
{md(med.round(2).reset_index())}

**Takeaway:** compare the group medians and overlapping distributions rather than treating a single low grade as a deterministic rule. First-semester grade and passed-unit counts show the academic separation visible in the figure; admission grade provides an earlier comparison. These are associations, with predictive value to be measured in cross-validation.

![Age and approval rate](../figures/04_age_and_approval_rate.png)

**Takeaway:** approved/enrolled units is a candidate academic progress feature. {int(enrolled.eq(0).sum())} training records have zero enrolled units and therefore an undefined ratio; keep them explicit and handle them inside the model pipeline. Zero semester grades should not automatically become missing values, because they may reflect no successful assessments. Age distributions overlap across outcomes, so age alone is not an individual risk decision rule.

## 6. Financial indicators
![Financial outcome proportions](../figures/03_financial_outcome_rates.png)

{chr(10).join(rate_lines)}

**Takeaway:** these are within-group outcome percentages, with sample counts shown to avoid confusing larger groups with higher risk. `financial_dropout_rates.csv` includes descriptive 95% Wilson intervals for dropout shares. Small groups are less precise. Financial status may be related to academic performance, background and recording time. These comparisons do not establish causality, independently adjusted effects or which support intervention would work.

## 7. Distributions, context and correlations
![Numeric distributions](../figures/05_numeric_distributions.png)

**Takeaway:** inspect the actual shapes, zero concentrations and tails before choosing transformations. Robust scaling or standardization may help linear/distance-based models, but fit transformations inside training folds. Do not assume normality or remove tail observations automatically.

![Numeric correlations](../figures/06_numeric_correlations.png)

The strongest absolute numeric-feature correlations are:
{md(pairs.head(5).drop(columns='abs_rho').round(3))}

**Takeaway:** related academic measurements can carry overlapping information; individual feature importance may be shared across correlated predictors. Spearman correlation is descriptive and does not establish causality. Nominal category codes and Target are excluded from this numeric matrix. Course-level sample sizes and dropout shares are exported in `course_dropout_rates.csv`; rare courses require caution and numeric course codes have no ordered meaning.

## 8. Modeling implications
Use the shared split, fold-local preprocessing, and fixed-estimator feature-group comparisons. Financial timing remains unverified.

## 9. Assumptions, limitations and useful additional data
This is descriptive analysis of a historical institutional dataset. The random split estimates an internal comparison; it does not demonstrate future-cohort or cross-university generalization. Date-stamped administrative records, cohort identifiers and dropout dates would help validate temporal availability and prevent including students whose dropout preceded an intended semester-end prediction. Household income, aid amounts, attendance, support-service usage and intervention outcomes would improve understanding but are not in this CSV. Do not infer that changing a correlated feature will change outcomes.

## 10. Conclusion
Academic and financial indicators provide testable hypotheses for the modeling team. The EDA establishes data structure, imbalance, descriptive group differences and preparation needs. Whether one group predicts better, or adds value to the other, remains a cross-validated modeling question.

## Reference
Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict Students’ Dropout and Academic Success* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89. Dataset licensed CC BY 4.0. Variable meanings and label horizon: {SOURCE}.
'''
(ROOT/'reports/partb_eda_report.md').write_text(report)
print(json.dumps(metadata,indent=2))
print('\nACADEMIC MEDIANS\n',med.to_string())
print('\nFINANCIAL RATES\n',fin.to_string(index=False))


## 4. Preprocessing and feature engineering

Four original-column feature sets (18/24/21/27) exclude second-semester and macroeconomic fields. First-semester scenarios add two ratios and a zero-enrollment indicator. Learned imputation, encoding, and scaling remain inside each model pipeline. Financial availability dates remain unverified.


In [ ]:
# First-semester features
first_sem_cols = [
    col for col in df.columns
    if "1st sem" in col
]

# Second-semester features
second_sem_cols = [
    col for col in df.columns
    if "2nd sem" in col
]

print("First-semester features:", len(first_sem_cols))
print(first_sem_cols)

print("\nSecond-semester features:", len(second_sem_cols))
print(second_sem_cols)


In [ ]:
# Conditional financial indicators
financial_cols = [
    "Scholarship holder",
    "Debtor",
    "Tuition fees up to date"
]

# Macroeconomic context variables
macro_cols = [
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

# Core admission features
# Financial timing is not yet verified.
admission_cols = [
    col for col in df.columns
    if col not in first_sem_cols
    + second_sem_cols
    + financial_cols
    + macro_cols
    + ["Target"]
]

# Admission + first-semester academic features
first_semester_cols = admission_cols + first_sem_cols

# Exploratory financial scenarios
admission_financial_cols = admission_cols + financial_cols

first_semester_financial_cols = (
    first_semester_cols + financial_cols
)

# Feature sets for modeling
feature_sets = {
    "Admission Only": admission_cols,
    "First Semester": first_semester_cols,
    "Admission + Financial": admission_financial_cols,
    "First Semester + Financial": first_semester_financial_cols
}

for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")

# Ensure no future information or target is included
for cols in feature_sets.values():
    assert "Target" not in cols
    assert not set(second_sem_cols).intersection(cols)
X_train_admission = X_train[admission_cols]
X_train_first_sem = X_train[first_semester_cols]


In [ ]:
def add_first_semester_features(X):
    X = X.copy()

    enrolled = X["Curricular units 1st sem (enrolled)"]

    # Avoid division by zero
    denominator = enrolled.replace(0, np.nan)

    # Proportion of enrolled courses approved
    X["First semester pass rate"] = (
        X["Curricular units 1st sem (approved)"]
        / denominator
    )

    # Average evaluations per enrolled course
    X["Evaluations per enrolled course"] = (
        X["Curricular units 1st sem (evaluations)"]
        / denominator
    )

    # Preserve information about zero enrollment
    X["No first-semester enrollment"] = (
        enrolled == 0
    ).astype(int)

    return X


In [ ]:
sample = add_first_semester_features(
    X_train_first_sem.head(8)
)

display(sample[[
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (approved)",
    "First semester pass rate",
    "Evaluations per enrolled course",
    "No first-semester enrollment"
]])


In [ ]:
# Define Categorical Variables

categorical_features = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]


In [ ]:
# Build Preprocessing Pipeline

def make_preprocessor(columns, add_engineered=False):

    columns = list(columns)

    if add_engineered:
        if not set(first_sem_cols).issubset(columns):
            raise ValueError(
                "First-semester features are required for engineering."
            )

    # Identify categorical and numerical features
    cat_cols = [
        col for col in columns
        if col in categorical_features
    ]

    num_cols = [
        col for col in columns
        if col not in categorical_features
    ]

    if add_engineered:
        num_cols += [
            "First semester pass rate",
            "Evaluations per enrolled course",
            "No first-semester enrollment"
        ]

    # Numerical preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine transformations
    column_transformer = ColumnTransformer([
        ("numeric", numeric_pipeline, num_cols),
        ("categorical", categorical_pipeline, cat_cols)
    ])

    # Include feature engineering inside the pipeline
    if add_engineered:
        preprocessor = Pipeline([
            (
                "feature_engineering",
                FunctionTransformer(
                    add_first_semester_features,
                    validate=False
                )
            ),
            ("preprocessing", column_transformer)
        ])
    else:
        preprocessor = Pipeline([
            ("preprocessing", column_transformer)
        ])

    return preprocessor


In [ ]:
# Create preprocessing pipelines for all feature sets
preprocessors = {}

for name, cols in feature_sets.items():

    use_first_sem = "First Semester" in name

    preprocessors[name] = make_preprocessor(
        cols,
        add_engineered=use_first_sem
    )

# Keep original variable names for compatibility
admission_preprocessor = preprocessors["Admission Only"]

first_semester_preprocessor = preprocessors["First Semester"]

print("All preprocessing pipelines created successfully!")


In [ ]:
# Test whether the preprocessing pipeline works with a classifier

from sklearn.base import clone
smoke_model = Pipeline([
    ("preprocessing", clone(first_semester_preprocessor)),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fit using training data only
smoke_model.fit(X_train_first_sem, y_train)

print("Pipeline fitted successfully")

# Check transformed training data shape
X_train_transformed = smoke_model.named_steps[
    "preprocessing"
].transform(X_train_first_sem)

print("Transformed shape:", X_train_transformed.shape)


In [ ]:
# Check that selected feature groups contain expected columns
assert set(admission_cols).issubset(first_semester_cols)

# Check that training and test sets are separate
assert X_train.index.intersection(X_test.index).empty

# Check that no infinite values appear after preprocessing
from scipy import sparse

# Use the fitted preprocessor from the smoke-test model
transformed = smoke_model.named_steps[
    "preprocessing"
].transform(X_train_first_sem)

values = (
    transformed.data
    if sparse.issparse(transformed)
    else transformed
)

assert np.isfinite(values).all()

print("All preprocessing checks passed!")


In [ ]:
# Check that target is excluded
assert "Target" not in admission_cols
assert "Target" not in first_semester_cols

# Check that second-semester information is excluded
assert not set(second_sem_cols).intersection(admission_cols)
assert not set(second_sem_cols).intersection(first_semester_cols)

# Check feature groups
print("Admission-only features:", len(admission_cols))
print("First-semester features:", len(first_semester_cols))

# Check input dimensions
print("Admission training data:", X_train_admission.shape)
print("First-semester training data:", X_train_first_sem.shape)

print("\nAll passed")


In [ ]:
from sklearn.base import clone

# Check all feature sets and pipelines
for name, cols in feature_sets.items():

    X_subset = X_train[cols]

    # Use an independent copy of the pipeline
    pipeline = clone(preprocessors[name])

    # Fit preprocessing on training data only
    transformed = pipeline.fit_transform(X_subset)

    # Check the transformed values
    from scipy import sparse

    values = (
        transformed.data
        if sparse.issparse(transformed)
        else transformed
    )

    assert np.isfinite(values).all()

    print(
        f"{name}: {X_subset.shape[1]} original features, "
        f"{transformed.shape[1]} transformed features"
    )

print("\nAll preprocessing integration checks passed!")


## 5. Baselines and initial five-model comparison

Scores use five stratified training-data folds. In-sample training scores are retained to assess overfitting. The majority baseline and model comparison share the same folds.


In [ ]:
RANDOM_STATE = 42
cv = CV
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
class_names = label_encoder.classes_
dropout_class = int(label_encoder.transform(["Dropout"])[0])

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "dropout_recall": make_scorer(recall_score, labels=[dropout_class], average="macro", zero_division=0),
    "dropout_precision": make_scorer(precision_score, labels=[dropout_class], average="macro", zero_division=0)
}

baseline_cv = cross_validate(
    DummyClassifier(strategy="most_frequent"),
    X_train[feature_sets["Admission Only"]],
    y_train_encoded,
    cv=cv,
    scoring=scoring,
    n_jobs=2
)

baseline_results = {
    metric: {"mean": float(np.mean(baseline_cv[f"test_{metric}"])), "std": float(np.std(baseline_cv[f"test_{metric}"]))}
    for metric in scoring
}

baseline_table = pd.DataFrame(baseline_results).T
baseline_table.columns = ["CV Mean", "CV Std"]
display(baseline_table.round(4))


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=1, tree_method="hist", eval_metric="mlogloss")
}

comparison_rows = []

for feature_set_name, selected_columns in feature_sets.items():
    X_selected = X_train[selected_columns]
    for model_name, estimator in models.items():
        pipeline = Pipeline([
            ("preprocessing", clone(preprocessors[feature_set_name])),
            ("model", clone(estimator))
        ])
        scores = cross_validate(
            pipeline,
            X_selected,
            y_train_encoded,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=2,
            error_score="raise"
        )
        row = {"Feature Set": feature_set_name, "Model": model_name, "Number of Features": len(selected_columns)}
        for metric in scoring:
            row[f"CV {metric} Mean"] = float(np.mean(scores[f"test_{metric}"]))
            row[f"CV {metric} Std"] = float(np.std(scores[f"test_{metric}"]))
        row["Train F1 Macro Mean"] = float(np.mean(scores["train_f1_macro"]))
        row["F1 Generalization Gap"] = row["Train F1 Macro Mean"] - row["CV f1_macro Mean"]
        row["CV Fit Time Mean"] = float(np.mean(scores["fit_time"]))
        comparison_rows.append(row)
        print(f"{feature_set_name} | {model_name} | F1 macro: {row['CV f1_macro Mean']:.4f} ± {row['CV f1_macro Std']:.4f} | Dropout recall: {row['CV dropout_recall Mean']:.4f}")

comparison_results = pd.DataFrame(comparison_rows).sort_values(
    ["Feature Set", "CV f1_macro Mean", "CV dropout_recall Mean"],
    ascending=[True, False, False]
).reset_index(drop=True)

display(comparison_results.round(4))


In [ ]:
results_dir = RESULTS_ROOT / "part_d_results"
results_dir.mkdir(parents=True, exist_ok=True)
comparison_results.to_csv(results_dir / "model_cv_comparison.csv", index=False)
baseline_table.to_csv(results_dir / "majority_class_baseline.csv")

primary_sets = ["Admission Only", "First Semester"]
metric_plot = comparison_results[comparison_results["Feature Set"].isin(primary_sets)].pivot(
    index="Model", columns="Feature Set", values="CV f1_macro Mean"
).reindex(index=list(models), columns=primary_sets)

ax = metric_plot.plot(kind="bar", figsize=(12, 6), width=0.78)
ax.axhline(baseline_results["f1_macro"]["mean"], color="black", linestyle="--", label="Majority baseline")
ax.set_ylabel("5-fold CV Macro F1")
ax.set_xlabel("Model")
ax.set_title("Model Comparison: Admission Only vs First Semester")
ax.set_ylim(0, 1)
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(results_dir / "model_comparison_macro_f1.png", dpi=200, bbox_inches="tight")
plt.show()

recall_plot = comparison_results[comparison_results["Feature Set"].isin(primary_sets)].pivot(
    index="Model", columns="Feature Set", values="CV dropout_recall Mean"
).reindex(index=list(models), columns=primary_sets)

ax = recall_plot.plot(kind="bar", figsize=(12, 6), width=0.78)
ax.axhline(baseline_results["dropout_recall"]["mean"], color="black", linestyle="--", label="Majority baseline")
ax.set_ylabel("5-fold CV Dropout Recall")
ax.set_xlabel("Model")
ax.set_title("Dropout Detection: Admission Only vs First Semester")
ax.set_ylim(0, 1)
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(results_dir / "model_comparison_dropout_recall.png", dpi=200, bbox_inches="tight")
plt.show()

f1_scenarios = comparison_results.pivot(index="Model", columns="Feature Set", values="CV f1_macro Mean")
scenario_improvement = pd.DataFrame({
    "First Semester - Admission Only": f1_scenarios["First Semester"] - f1_scenarios["Admission Only"],
    "Admission + Financial - Admission Only": f1_scenarios["Admission + Financial"] - f1_scenarios["Admission Only"],
    "First Semester + Financial - First Semester": f1_scenarios["First Semester + Financial"] - f1_scenarios["First Semester"]
}).sort_values("First Semester - Admission Only", ascending=False)
scenario_improvement.to_csv(results_dir / "feature_set_improvements.csv")
display(scenario_improvement.round(4))


### Shortlist and preliminary out-of-fold error analysis

The shortlist is ranked by first-semester macro F1. The following confusion matrix belongs to the best initial untuned model, not the later selected ensemble. All predictions are out of fold on training rows.


In [ ]:
selection_feature_set = "First Semester"
shortlist_table = comparison_results[comparison_results["Feature Set"] == selection_feature_set].sort_values(
    ["CV f1_macro Mean", "CV dropout_recall Mean", "CV f1_macro Std"],
    ascending=[False, False, True]
).head(3).reset_index(drop=True)

shortlisted_models = shortlist_table["Model"].tolist()
shortlisted_pipelines = {
    name: Pipeline([
        ("preprocessing", clone(preprocessors[selection_feature_set])),
        ("model", clone(models[name]))
    ])
    for name in shortlisted_models
}

shortlist_table.to_csv(results_dir / "top_three_models.csv", index=False)
display(shortlist_table.round(4))

trained_shortlist = {
    name: clone(pipeline).fit(X_train[feature_sets[selection_feature_set]], y_train_encoded)
    for name, pipeline in shortlisted_pipelines.items()
}
print("Shortlisted models:", shortlisted_models)


In [ ]:
oof_predictions = {}
oof_metrics = []

def oof_report(model_name, predictions):
    report = classification_report(
        y_train_encoded,
        predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    for class_name in class_names:
        oof_metrics.append({
            "Model": model_name,
            "Class": class_name,
            "Precision": report[class_name]["precision"],
            "Recall": report[class_name]["recall"],
            "F1": report[class_name]["f1-score"],
            "Support": int(report[class_name]["support"])
        })

for model_name, pipeline in shortlisted_pipelines.items():
    predictions = cross_val_predict(
        pipeline,
        X_train[feature_sets[selection_feature_set]],
        y_train_encoded,
        cv=cv,
        n_jobs=2
    )
    oof_predictions[model_name] = predictions
    oof_report(model_name, predictions)

cv_class_performance = pd.DataFrame(oof_metrics)
cv_class_performance.to_csv(results_dir / "shortlist_oof_class_performance.csv", index=False)
display(cv_class_performance.round(4))


In [ ]:
best_model_name = shortlisted_models[0]
best_oof_predictions = oof_predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, normalization, title, value_format in [
    (axes[0], None, "Counts", "d"),
    (axes[1], "true", "Row-Normalized", ".2f")
]:
    cm = confusion_matrix(
        y_train_encoded,
        best_oof_predictions,
        labels=np.arange(len(class_names)),
        normalize=normalization
    )
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=value_format
    )
    ax.set_title(f"{best_model_name}: {title}")

plt.tight_layout()
plt.savefig(results_dir / "best_model_cv_confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

error_table = pd.DataFrame({
    "Actual": label_encoder.inverse_transform(y_train_encoded),
    "Predicted": label_encoder.inverse_transform(best_oof_predictions)
})

error_counts = (
    error_table.loc[error_table["Actual"] != error_table["Predicted"]]
    .groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
error_counts["Share of All Errors"] = error_counts["Count"] / error_counts["Count"].sum()
error_counts.to_csv(results_dir / "best_model_cv_error_types.csv", index=False)
display(error_counts.round(4))


## 6. Tune the shortlisted models

Randomized search reuses the training CV folds (10 trials each for boosting models and 8 for Logistic Regression in full mode). This is not nested CV; reported selection metrics can be optimistic.


In [ ]:
PRIMARY_SET = selection_feature_set
X_primary = X_train[feature_sets[PRIMARY_SET]]
dropout_code = dropout_class
assert set(shortlisted_models) == {'XGBoost', 'Gradient Boosting', 'Logistic Regression'}


In [ ]:
models = {
    'XGBoost': XGBClassifier(n_estimators=150, random_state=RANDOM_STATE,
                             n_jobs=1, tree_method='hist', eval_metric='mlogloss'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
}
parameter_distributions = {
    'XGBoost': {
        'model__n_estimators': randint(80, 261),
        'model__max_depth': randint(2, 6),
        'model__learning_rate': loguniform(0.02, 0.20),
        'model__subsample': uniform(0.65, 0.35),
        'model__colsample_bytree': uniform(0.65, 0.35),
        'model__min_child_weight': randint(1, 9),
        'model__reg_lambda': loguniform(0.3, 30.0),
    },
    'Gradient Boosting': {
        'model__n_estimators': randint(80, 261),
        'model__max_depth': randint(1, 4),
        'model__learning_rate': loguniform(0.02, 0.20),
        'model__subsample': uniform(0.65, 0.35),
        'model__min_samples_leaf': randint(1, 16),
    },
    'Logistic Regression': {
        'model__C': loguniform(0.03, 30.0),
        'model__class_weight': [None, 'balanced'],
    },
}
searches = {}
search_rows = []
for model_name, estimator in models.items():
    pipe = Pipeline([('preprocessing', clone(preprocessors[PRIMARY_SET])),
                     ('model', clone(estimator))])
    search = RandomizedSearchCV(
        estimator=pipe, param_distributions=parameter_distributions[model_name],
        n_iter=ITERATIONS[model_name], cv=CV, scoring='f1_macro', refit=True,
        n_jobs=1, random_state=RANDOM_STATE, error_score='raise',
        return_train_score=True
    )
    print(f'Tuning {model_name} ({ITERATIONS[model_name]} samples x {N_SPLITS} folds) ...')
    search.fit(X_primary, y_train_encoded)
    searches[model_name] = search
    idx = search.best_index_
    search_rows.append({
        'Model': model_name,
        'CV Macro F1 Mean': float(search.cv_results_['mean_test_score'][idx]),
        'CV Macro F1 Std': float(search.cv_results_['std_test_score'][idx]),
        'Train Macro F1 Mean': float(search.cv_results_['mean_train_score'][idx]),
        'Train - CV Gap': float(search.cv_results_['mean_train_score'][idx] -
                               search.cv_results_['mean_test_score'][idx]),
        'Best Params': json.dumps(search.best_params_, sort_keys=True)
    })
    pd.DataFrame(search.cv_results_).to_csv(
        OUTPUT_DIR / f'search_trials_{model_name.lower().replace(" ", "_")}.csv', index=False)

tuning_results = pd.DataFrame(search_rows).sort_values(
    'CV Macro F1 Mean', ascending=False).reset_index(drop=True)
tuning_results.to_csv(OUTPUT_DIR / 'tuning_comparison.csv', index=False)
display(tuning_results[['Model', 'CV Macro F1 Mean', 'CV Macro F1 Std',
                        'Train - CV Gap']].round(4))


## 7. Tuned candidates and equal-weight soft voting

Each voting member contains its own full preprocessing pipeline. No final test predictions are generated.


In [ ]:
def summarize_cv(name, estimator, features, labels):
    result = cross_validate(estimator, features, labels, cv=CV, scoring=scoring,
                            n_jobs=1, error_score='raise')
    return {
        'Candidate': name,
        'Macro F1 Mean': float(np.mean(result['test_f1_macro'])),
        'Macro F1 Std': float(np.std(result['test_f1_macro'])),
        'Dropout Recall Mean': float(np.mean(result['test_dropout_recall'])),
        'Dropout Recall Std': float(np.std(result['test_dropout_recall'])),
        'Accuracy Mean': float(np.mean(result['test_accuracy'])),
    }

# Fit each voting member through its own cloneable full preprocessing pipeline.
tuned_pipelines = {name: clone(search.best_estimator_)
                   for name, search in searches.items()}
soft_vote = VotingClassifier(
    estimators=[('xgb', clone(tuned_pipelines['XGBoost'])),
                ('gb', clone(tuned_pipelines['Gradient Boosting'])),
                ('lr', clone(tuned_pipelines['Logistic Regression']))],
    voting='soft', n_jobs=1)

candidate_rows = [summarize_cv(n, model, X_primary, y_train_encoded)
                  for n, model in tuned_pipelines.items()]
candidate_rows.append(summarize_cv('Soft Voting Ensemble', soft_vote,
                                   X_primary, y_train_encoded))
candidate_results = pd.DataFrame(candidate_rows).sort_values(
    ['Macro F1 Mean', 'Dropout Recall Mean'], ascending=False).reset_index(drop=True)
candidate_results.to_csv(OUTPUT_DIR / 'candidate_ensemble_cv.csv', index=False)
display(candidate_results.round(4))

WINNER_NAME = str(candidate_results.iloc[0]['Candidate'])
WINNER_PIPELINE = (soft_vote if WINNER_NAME == 'Soft Voting Ensemble'
                   else tuned_pipelines[WINNER_NAME])
print('Provisional CV winner; independent evaluation remains required:', WINNER_NAME)

plt.figure(figsize=(9.4, 4.8))
plot = candidate_results.sort_values('Macro F1 Mean')
plt.barh(plot['Candidate'], plot['Macro F1 Mean'], xerr=plot['Macro F1 Std'],
         capsize=3, color='#2476a2')
plt.xlabel('Macro F1 (5-fold training CV; ± fold SD)')
plt.title('Tuned candidates vs. soft-voting ensemble')
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_tuning_and_ensemble.png', dpi=180, bbox_inches='tight')
plt.show()


## 8. Fixed-model academic and financial feature comparison

This reruns untuned Logistic Regression under identical conditions across feature sets. Historical saved outputs differ slightly for Admission + Financial (initial comparison 0.5461 versus later ablation 0.5464). Both calculations are retained below and their new-run difference is exported; no historical result is silently substituted.


In [ ]:
ablation_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
ablation_rows = []
for set_name, cols in feature_sets.items():
    pipeline = Pipeline([('preprocessing', clone(preprocessors[set_name])),
                         ('model', clone(ablation_model))])
    row = summarize_cv(set_name, pipeline, X_train[cols], y_train_encoded)
    ablation_rows.append({'Feature Set': set_name, 'Raw Feature Count': len(cols),
                          **{k: v for k, v in row.items() if k != 'Candidate'}})
ablation_results = pd.DataFrame(ablation_rows).set_index('Feature Set').reindex(
    list(feature_sets)).reset_index()
ablation_results.to_csv(OUTPUT_DIR / 'academic_financial_ablation.csv', index=False)
display(ablation_results.round(4))

f1_by_set = ablation_results.set_index('Feature Set')['Macro F1 Mean']
academic_gain = f1_by_set['First Semester'] - f1_by_set['Admission Only']
financial_gain_admission = f1_by_set['Admission + Financial'] - f1_by_set['Admission Only']
financial_gain_semester = (f1_by_set['First Semester + Financial'] -
                           f1_by_set['First Semester'])
print(f'Additional first-semester academic measurements: Δ Macro F1 = {academic_gain:+.4f}')
print(f'Financial indicators at admission (conditional): Δ Macro F1 = {financial_gain_admission:+.4f}')
print(f'Financial indicators after semester 1 (conditional): Δ Macro F1 = {financial_gain_semester:+.4f}')

plt.figure(figsize=(10.2, 5.2))
plt.barh(ablation_results['Feature Set'], ablation_results['Macro F1 Mean'],
         xerr=ablation_results['Macro F1 Std'], color='#277e7e', capsize=3)
plt.xlim(0, 1)
plt.xlabel('Macro F1 (5-fold CV; ± fold SD)')
plt.title('Incremental predictive value of academic and financial inputs')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_feature_group_ablation.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
initial_lr = comparison_results.query("Model == 'Logistic Regression'").set_index('Feature Set')
repeated_lr = ablation_results.set_index('Feature Set')
reconciliation = pd.DataFrame({
    'Initial Macro F1': initial_lr['CV f1_macro Mean'],
    'Repeated Macro F1': repeated_lr['Macro F1 Mean'],
    'Initial Dropout Recall': initial_lr['CV dropout_recall Mean'],
    'Repeated Dropout Recall': repeated_lr['Dropout Recall Mean'],
})
reconciliation['Macro F1 Difference'] = reconciliation['Repeated Macro F1'] - reconciliation['Initial Macro F1']
reconciliation.to_csv(OUTPUT_DIR / 'run_reconciliation.csv')
display(reconciliation)


## 9. Permutation importance

Importance is measured on validation folds within training data. The primary explanation uses the best tuned single model; the conditional financial explanation uses untuned Logistic Regression. Neither directly explains the ensemble or establishes causal effects. The selected hyperparameters already saw these folds during tuning.


In [ ]:
single_model_name = str(tuning_results.iloc[0]['Model'])
single_estimator = clone(tuned_pipelines[single_model_name])

def cv_permutation_importance(pipeline, set_name, repeats=N_PERM_REPEATS):
    columns = list(feature_sets[set_name])
    X_set = X_train[columns]
    rows = []
    for fold, (train_idx, val_idx) in enumerate(CV.split(X_set, y_train_encoded), start=1):
        fitted = clone(pipeline).fit(X_set.iloc[train_idx], y_train_encoded[train_idx])
        importance = permutation_importance(
            fitted, X_set.iloc[val_idx], y_train_encoded[val_idx],
            scoring='f1_macro', n_repeats=repeats, random_state=RANDOM_STATE + fold,
            n_jobs=1)
        for feat, vals in zip(columns, importance.importances):
            rows.extend({'Feature': feat, 'Fold': fold, 'Importance': float(x)} for x in vals)
        print(f'  {set_name}: finished fold {fold}/{N_SPLITS}')
    raw = pd.DataFrame(rows)
    summary = (raw.groupby('Feature')['Importance']
               .agg(Mean='mean', Std='std').sort_values('Mean', ascending=False)
               .reset_index())
    return raw, summary

print('Permutation importance for best single tuned model:', single_model_name)
primary_perm_raw, primary_importance = cv_permutation_importance(single_estimator, PRIMARY_SET)
primary_perm_raw.to_csv(OUTPUT_DIR / 'perm_primary_fold_values.csv', index=False)
primary_importance.to_csv(OUTPUT_DIR / 'perm_primary_summary.csv', index=False)
display(primary_importance.head(15).round(4))

def plot_importance(table, title, filename, top=12):
    subset = table.head(top).iloc[::-1]
    plt.figure(figsize=(10.2, max(4, top * 0.42 + 1.2)))
    plt.barh(subset['Feature'], subset['Mean'], xerr=subset['Std'].fillna(0),
             color='#8463ac', capsize=2)
    plt.axvline(0, color='#555', linewidth=0.8)
    plt.xlabel('Mean decrease in validation Macro F1 after permutation')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()

plot_importance(primary_importance, f'First-semester predictors — {single_model_name}',
                '03_primary_feature_importance.png')


In [ ]:
FULL_SET = 'First Semester + Financial'
financial_pipeline = Pipeline([
    ('preprocessing', clone(preprocessors[FULL_SET])),
    ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
finance_perm_raw, finance_importance = cv_permutation_importance(financial_pipeline, FULL_SET)
finance_perm_raw.to_csv(OUTPUT_DIR / 'perm_financial_fold_values.csv', index=False)
finance_importance.to_csv(OUTPUT_DIR / 'perm_financial_summary.csv', index=False)
display(finance_importance.head(15).round(4))

plot_importance(finance_importance, 'First semester + financial (conditional) — Logistic Regression',
                '04_conditional_financial_importance.png')
print('Financial feature ranks:')
display(finance_importance.assign(Rank=np.arange(1, len(finance_importance) + 1))
        .query('Feature in @financial_cols')[['Feature', 'Rank', 'Mean', 'Std']].round(4))


## 10. Export report, selected configuration, and run metadata

This exports a configuration for later final evaluation, not a final test result or serialized model. Financial timing is unverified and Enrolled does not settle eventual outcomes.


In [ ]:
top_features = primary_importance.head(8)['Feature'].tolist()
top_financial = finance_importance[finance_importance['Feature'].isin(financial_cols)]

report = f"""# Tuning — Hyperparameter Tuning and Feature Importance

## Objective and reproducibility
We build on the shared stratified 80/20 split (seed 42), 5-fold stratified CV, and four predeclared feature sets. Only training rows are used in this part. We compare models by macro F1, with dropout recall as an operational secondary metric. The final test set is reserved for final evaluation.

## Hyperparameter tuning (First Semester, no financial indicators)
We applied RandomizedSearchCV with five stratified folds to the three models shortlisted above: XGBoost, Gradient Boosting, and Logistic Regression. Search spaces addressed model complexity, learning rate, regularization, and (for logistic regression) class weighting. This is a randomized search of a bounded hyperparameter space, not an exhaustive optimization or a causal experiment.

{tuning_results[['Model', 'CV Macro F1 Mean', 'CV Macro F1 Std', 'Train - CV Gap']].round(4).to_markdown(index=False)}

Figure: ![Tuned model and ensemble comparison](01_tuning_and_ensemble.png)

## Soft-voting ensemble and model selection
A three-model soft-voting ensemble was compared with each tuned model using the same first-semester inputs and five-fold CV.

{candidate_results.round(4).to_markdown(index=False)}

**Provisional training-CV choice:** {WINNER_NAME}. These cross-validation results help with selection but may be optimistic because the same training data informs tuning and selection. final evaluation must evaluate the locked model once on the untouched held-out test set. No final-test performance claim is made here.

## Incremental academic and financial information
To hold modeling conditions constant, we compared all four feature sets using the same untuned Logistic Regression estimator and the same five stratified folds:

{ablation_results[['Feature Set','Raw Feature Count','Macro F1 Mean','Macro F1 Std','Dropout Recall Mean']].round(4).to_markdown(index=False)}

- Adding first-semester academic results to admission context: Δ macro F1 = **{academic_gain:+.4f}**.
- Adding financial variables to admission context: Δ macro F1 = **{financial_gain_admission:+.4f}** (conditional on verified timing).
- Adding financial variables to first-semester context: Δ macro F1 = **{financial_gain_semester:+.4f}** (conditional on verified timing).

![Feature-set CV comparison](02_feature_group_ablation.png)

This is a comparison of predictive contributions under a fixed-model CV setup. The admission group already includes some academic-background attributes, so the academic delta specifically measures **added first-semester observations**. The financial fields (scholarship, debtor, tuition-payment status) have unverified recording times. Do not interpret these associations as causal effects or proven deployable early-warning inputs.

## Feature importance
For interpretability we computed permutation importance on training-held-out CV folds using the best tuned single model ({single_model_name}) and reported the reduction in validation macro F1 when an original column was shuffled. The top eight columns were: **{', '.join(top_features)}**.

![First-semester permutation importance](03_primary_feature_importance.png)

A separate conditional model using First Semester + Financial was interpreted to examine the three finance-related columns:

{top_financial.round(4).to_markdown(index=False)}

![Financial-scenario permutation importance](04_conditional_financial_importance.png)

Permutation importance is model-specific and may distribute importance across correlated inputs; it does not imply causality or justify an individual-level intervention by itself.

## Limitations and next step for final evaluation
- Hyperparameter selection and model comparison reuse the training folds; reported CV results are exploratory model-selection metrics and potentially optimistic.
- The Enrolled label does not settle students' eventual outcomes. Its class-specific performance should be explicitly reported at final evaluation.
- Financial administrative timestamps are unverified. The **main** selected first-semester modeling scenario intentionally excludes the financial variables.
- No held-out test scores were calculated here. The final evaluation should take the frozen selection from `model_selection_for_F.json`, reproduce the training pipeline, evaluate once, and discuss confusion matrices, recall/F1, and realistic limitations.
- Before finalizing, the team should verify all rubric items and ensure the repository contains final documentation and evidence figures.
"""
(OUTPUT_DIR / 'PART_E_REPORT.md').write_text(report, encoding='utf-8')

params = {name: s.best_params_ for name, s in searches.items()}
handoff = {
    'selection_feature_set': PRIMARY_SET,
    'feature_columns': list(feature_sets[PRIMARY_SET]),
    'chosen_candidate': WINNER_NAME,
    'tuned_parameters': params,
    'train_test_split': {'test_size': 0.2, 'stratify': 'Target', 'random_state': RANDOM_STATE},
    'cv': {'n_splits': N_SPLITS, 'shuffle': True, 'random_state': RANDOM_STATE},
    'primary_selection_metric': 'macro F1',
    'important_note': 'CV selection only. Held-out test set NOT accessed in Tuning.',
    'financial_features_timing_verified': False,
    'fast_mode': FAST_MODE,
}
(OUTPUT_DIR / 'model_selection_for_F.json').write_text(
    json.dumps(handoff, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved', OUTPUT_DIR / 'PART_E_REPORT.md')
print('Saved', OUTPUT_DIR / 'model_selection_for_F.json')
print('Saved CSV and figures in', OUTPUT_DIR.resolve())


In [ ]:
import platform
import importlib.metadata
packages = ['pandas', 'numpy', 'scipy', 'scikit-learn', 'xgboost', 'matplotlib', 'seaborn', 'ipython', 'tabulate']
versions = {}
for package in packages:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = 'not installed'
run_metadata = {
    'python': platform.python_version(), 'packages': versions,
    'source': DATA_SOURCE, 'data_sha256': hashlib.sha256(DATA_BYTES).hexdigest(),
    'train_n': len(train_ids), 'test_n': len(test_ids),
    'seed': RANDOM_STATE, 'fast_mode': FAST_MODE,
    'test_performance_calculated': False,
}
(RESULTS_ROOT / 'environment_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print('Outputs:', RESULTS_ROOT)
print('Reserved test set has not been scored. Final evaluation remains separate.')


## Sources and interpretation limits

Source files: `insy674_partb_result/PartB_EDA_Script.py`, `Part C.ipynb`, `Part_D.ipynb`, and `Part E.ipynb` in chuyu-xu/INSY674-Student-Dropout-Prediction (main, retrieved 2026-10-28). Original algorithms, search spaces, seed, metrics, and exported result filenames are retained; repeated loading, splitting, and preprocessing definitions are consolidated. Outputs are isolated under `combined_results/` so original evidence is preserved.

Financial timestamps and other input availability assumptions need confirmation. Training CV does not establish future-cohort performance, fairness, calibration, or intervention effectiveness. Final selected-model test metrics and error analysis belong to subsequent final evaluation.

Dataset: Realinho et al. (2021), *Predict Students’ Dropout and Academic Success*, UCI Machine Learning Repository, https://doi.org/10.24432/C5MC89 (CC BY 4.0).
